# M3 CLV 조건부 방향성 first-hop 그래프 - Dunnhumby

이 노트북은 **기존 이진 M1 그래프와 2-hop 협업 신호를 보존**하면서, 사용자 최종 표현의 1-hop 항 하나에만 historical CLV proxy를 반영하는 빠른 탐색 실험입니다.

- historical CLV proxy: `N_hat × V_hat` (`N_hat`: train 장바구니 수, `V_hat`: train 평균 장바구니 금액)
- 관계값: 한 상품이 그 사용자의 장바구니 금액에서 차지한 평균 비중을 사용자 내부에서 순위화한 값
- 실제 CLV arm: `exp(beta × q_CLV(user) × relation(user,item))`
- 사용자별 M1 first-hop 메시지 총량은 정확히 보존
- M1의 상품 방향, 2-hop 표현, 최종 상품 표현은 변경하지 않음

비교 대상은 M1, CLV-free 관계-only, 실제 CLV, 사용자 이진 degree 10분위 내 CLV-shuffle입니다. 세 개 활성 arm의 개입 강도는 0.075로 맞춥니다. 주 판정은 실제 CLV의 Recall/NDCG @10·20·50 기하평균 균형이 세 비교군을 모두 넘는지입니다. 가격·구매금액 가중 적중값과 노출 지표는 설명용으로만 봅니다.

DAY 1--683을 학습하고 DAY 684--690을 탐색 평가합니다. 최종 test와 holdout은 만들지 않습니다. seed 42 한 번이므로 유의성이나 일반화를 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess

REVIEWED_SHA = '4aed34690230ad4bb5e1e6d8f261e99a640dacf2'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner
print('Pinned execution source:', actual_sha)

In [ ]:
import json, torch
from lightgcn_clv_m3_directional_first_hop import (
    configure_directional_first_hop_run,
    preflight_summary,
    run_directional_first_hop_screen,
)

cfg = configure_directional_first_hop_run()
assert torch.cuda.is_available(), 'Colab 런타임에서 GPU를 선택한 뒤 다시 실행하세요.'
summary = preflight_summary(cfg)
assert summary['seed'] == 42
assert summary['historical_development_split']['final_test_constructed'] is False
assert summary['historical_development_split']['holdout_constructed'] is False
assert summary['m3']['historical_clv_proxy'] == 'N_hat * V_hat'
assert summary['m3']['changed_term'] == 'user first-hop only'
assert summary['m3']['target_first_hop_strength'] == 0.075
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['sample_weighting'] is False
assert summary['fixed']['one_training_loop_and_optimizer'] is True
assert summary['fixed']['min_item_interactions'] == 1
assert summary['reading_rule']['accuracy_guardrails'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_directional_first_hop_screen(cfg)

In [ ]:
from IPython.display import display

columns = [
    'model_id', 'role',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'price_purchase_amount_weighted_hit@20',
    'price_purchase_amount_weighted_hit@50',
    'mean_recommended_price_percentile@10',
    'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10',
]
available = [column for column in columns if column in result_df.columns]
display(result_df[available])

print('\n실제 CLV 귀속 판정:')
print(json.dumps(result_df.attrs['attribution_reading'], ensure_ascii=False, indent=2))
print('\n개입 강도와 질량 보존 진단:')
print(json.dumps(result_df.attrs['graph_diagnostics']['arms'], ensure_ascii=False, indent=2))
print('\n결과 파일:', result_df.attrs['result_paths'])